In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

players = spark.table("lh_silver_game.players_clean")
sessions = spark.table("lh_silver_game.sessions_clean")

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 3, Finished, Available, Finished, False)

In [2]:
player_activity = (
    sessions
    .select(
        "player_id",
        "days_since_install"
    )
    .distinct()
)

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 4, Finished, Available, Finished, False)

In [3]:
retention_flags = (
    player_activity
    .groupBy("player_id")
    .agg(
        F.max(
            F.when(F.col("days_since_install") == 1, 1).otherwise(0)
        ).alias("d1_retained"),

        F.max(
            F.when(F.col("days_since_install") == 7, 1).otherwise(0)
        ).alias("d7_retained"),

        F.max(
            F.when(F.col("days_since_install") == 30, 1).otherwise(0)
        ).alias("d30_retained")
    )
)

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 5, Finished, Available, Finished, False)

In [4]:
player_retention = (
    players
    .select(
        "player_id",
        "install_date",
        "country",
        "platform",
        "acquisition_channel",
        "experiment_group",
        "creative_experiment_group"
    )
    .join(
        retention_flags,
        on="player_id",
        how="left"
    )
    .fillna({
        "d1_retained": 0,
        "d7_retained": 0,
        "d30_retained": 0
    })
)

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 6, Finished, Available, Finished, False)

In [5]:
print("Player count:", player_retention.count())

display(
    player_retention.select(
        "player_id",
        "install_date",
        "d1_retained",
        "d7_retained",
        "d30_retained"
    ).limit(10)
)

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 7, Finished, Available, Finished, False)

Player count: 50000


SynapseWidget(Synapse.DataFrame, 74298c91-7fa4-40e6-9c8e-a43088b4789a)

In [6]:
print("Player count:", player_retention.count())

display(
    player_retention.select(
        "player_id",
        "install_date",
        "d1_retained",
        "d7_retained",
        "d30_retained"
    ).limit(10)
)

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 8, Finished, Available, Finished, False)

Player count: 50000


SynapseWidget(Synapse.DataFrame, 896bd4d2-b2b3-4a7a-be3b-c893af17704b)

In [7]:
display(player_retention.limit(10))

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 716d6ef4-0d31-4bb7-97ef-a9fc6a45994d)

In [8]:
retention_cohorts = (
    player_retention
    .groupBy(
        "install_date",
        "acquisition_channel"
    )
    .agg(
        F.count("*").alias("cohort_size"),
        F.avg("d1_retained").alias("d1_retention"),
        F.avg("d7_retained").alias("d7_retention"),
        F.avg("d30_retained").alias("d30_retention")
    )
)

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 10, Finished, Available, Finished, False)

In [9]:
display(
    retention_cohorts
    .orderBy("install_date", "acquisition_channel")
    .limit(20)
)

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b1c294ab-8c95-47bd-92ae-15235247de8b)

In [10]:
retention_cohorts.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_gold_game.retention_cohorts")

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 12, Finished, Available, Finished, False)

In [11]:
df_check = spark.table("lh_gold_game.retention_cohorts")

print("Saved row count:", df_check.count())
display(df_check.limit(10))

StatementMeta(, a482284d-c744-450b-9f6e-8ba070a3e9d8, 13, Finished, Available, Finished, False)

Saved row count: 392


SynapseWidget(Synapse.DataFrame, 0a6250cf-5350-4399-8c28-d65ab441a298)